# [Step 5 - DirectoryLoader] Bulk-loading a folder tree

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

### What you'll learn

- wrapping any single-file loader with `DirectoryLoader` for bulk ingestion
- glob patterns + `recursive=True` for folder trees
- summarizing loaded Documents per source file
- per-file failure behavior: loud default vs `silent_errors=True` skips

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================

# --- Standard library -------------------------------------------------
import logging                  # making hidden skip-warnings VISIBLE below
import os                       # file cleanup for the pitfall demo
import urllib.request           # polite HTTP fetching
from pathlib import Path        # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# ---------------------------------------------------------------------
# TRACK WALKER - resolve 03_agentic_ai by walking upward from cwd.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - fetch once, cache under DATA, reuse forever.
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    target.write_bytes(payload)
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; drops a BOM if present."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe printing for arbitrary text."""
    return text.encode("ascii", errors="replace").decode("ascii")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_59236\4116454577.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


### 1. From one file to a whole tree

Real corpora are FOLDERS: hundreds of mixed files, nested subfolders.
`DirectoryLoader` is a wrapper, not a parser: you hand it

- a root `path`,
- a `glob` pattern selecting files (`"**/*.txt"` = every .txt anywhere below),
- a `loader_cls` (which single-file loader to instantiate per hit),
- `loader_kwargs` forwarded to that loader (here: our utf-8 encoding),

and it walks the tree (`recursive=True`!), loads each match, concatenates
all Documents. Each Document's `metadata["source"]` still points at its
own file - so provenance survives bulk loading.

We build a small but honest corpus under `data/notes/`: two course-authored
sample notes (clearly labeled), plus an excerpt copied from cached Alice.

In [2]:
# --- Ensure sibling datasets exist first (cache hits after notebook 01/04)
get_bytes("alice.txt", "https://www.gutenberg.org/files/11/11-0.txt")
get_bytes("posts.json", "https://jsonplaceholder.typicode.com/posts")

# --- Build data/notes -------------------------------------------------
notes_dir = DATA / "notes"
notes_dir.mkdir(exist_ok=True)

# Course-authored sample notes (allowed: clearly labeled teaching material).
(notes_dir / "study_tip.txt").write_text(
    "COURSE SAMPLE NOTE (authored by MLCourse).\n\n"
    "Read WHAT and WHY before code. Then retype the smallest example\n"
    "that proves the point to yourself. Understanding compresses;\n"
    "memorization does not.\n",
    encoding="utf-8",
)
(notes_dir / "glossary.txt").write_text(
    "COURSE SAMPLE NOTE (authored by MLCourse).\n\n"
    "Document = page_content + metadata.\n"
    "Loader = file(s) to Documents.\n"
    "Chunk = retrievable slice of one Document.\n",
    encoding="utf-8",
)

# Copy an excerpt from the cached Alice book into the folder too.
alice_raw = (DATA / "alice.txt").read_bytes().decode("utf-8-sig", errors="replace")
(notes_dir / "alice_excerpt.txt").write_text(alice_raw[:1200], encoding="utf-8")

for f in sorted(notes_dir.glob("*.txt")):
    print(f"{f.name:<22} {f.stat().st_size:>6,} bytes")

[cache] alice.txt: 151,191 bytes
[cache] posts.json: 27,520 bytes
alice_excerpt.txt       1,260 bytes
glossary.txt              159 bytes
study_tip.txt             197 bytes


### 3. Load the whole folder in one call

Note the two easy-to-miss arguments: `recursive=True` (the default glob
would otherwise only scan the top level with `rglob` semantics OFF) and
`loader_kwargs={"encoding": "utf-8"}` (TextLoader defaults would still be
fine here, but explicit beats implicit in batch jobs).

In [3]:
dir_loader = DirectoryLoader(
    str(notes_dir),
    glob="**/*.txt",                       # every .txt at ANY depth
    loader_cls=TextLoader,                 # same loader we mastered in nb 01
    loader_kwargs={"encoding": "utf-8"},   # forwarded to TextLoader
    recursive=True,                        # descend into subfolders
    show_progress=True,                    # tqdm bar over files
)
docs = dir_loader.load()

print(f"\nDocuments loaded : {len(docs)}")

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 162.06it/s]


Documents loaded : 3


### 4. Inspect + meaningful manipulation: summarize per source

Bulk loads deserve bulk audits: which file contributed how many Documents
and how much text? Group by `metadata["source"]`.

In [4]:
from collections import defaultdict
summary = defaultdict(lambda: {"docs": 0, "chars": 0})
for doc in docs:
    name = Path(doc.metadata["source"]).name
    summary[name]["docs"] += 1
    summary[name]["chars"] += len(doc.page_content)

print(f"{'file':<22}{'docs':>6}{'chars':>10}")
for name, stats in sorted(summary.items()):
    print(f"{name:<22}{stats['docs']:>6}{stats['chars']:>10,}")

print("\nsample content preview:")
print(to_ascii(docs[0].page_content[:150]))

file                    docs     chars
alice_excerpt.txt          1     1,200
glossary.txt               1       154
study_tip.txt              1       192

sample content preview:
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice?s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION


### 5. The silent-skip pitfall, demonstrated safely

One poisoned file can ruin a batch. Watch BOTH behaviors:

- default `silent_errors=False`: the FIRST bad file RAISES - load dies;
- `silent_errors=True`: bad files are skipped with only a logger warning -
  your run 'succeeds' with silently missing data. That is the trap.

We write a deliberately broken file (invalid UTF-8 byte 0xFF) and try both.

In [5]:
broken_path = notes_dir / "_broken_encoding.txt"
broken_path.write_bytes(b"Caf\xe9 na\xefve then an invalid byte: \xff")

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")

patient_loader = DirectoryLoader(
    str(notes_dir), glob="**/*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}, recursive=True,
    silent_errors=True,                     # <- skip-and-warn mode
)
survivors = patient_loader.load()
print(f"[silent_errors=True]  loaded {len(survivors)} docs "
      f"(expected {len(docs)} - the broken file was SKIPPED, see warning above)")

strict_loader = DirectoryLoader(
    str(notes_dir), glob="**/*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}, recursive=True,
    silent_errors=False,                    # <- default: fail loudly
)
try:
    strict_loader.load()
except Exception as err:
    print(f"[silent_errors=False] raised as expected: "
          f"{type(err).__name__}: {str(err)[:80]}...")
finally:
    os.remove(broken_path)                  # clean up the poison pill
    print("\n[cleanup] broken demo file removed.")

post_cleanup = DirectoryLoader(
    str(notes_dir), glob="**/*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}, recursive=True,
).load()
print(f"[after cleanup] back to {len(post_cleanup)} clean Documents.")

WARNING Error loading file D:\projects\python\MLCourse\03_agentic_ai\data\notes\_broken_encoding.txt: Error loading D:\projects\python\MLCourse\03_agentic_ai\data\notes\_broken_encoding.txt


ERROR Error loading file D:\projects\python\MLCourse\03_agentic_ai\data\notes\_broken_encoding.txt


[silent_errors=True]  loaded 3 docs (expected 3 - the broken file was SKIPPED, see warning above)
[silent_errors=False] raised as expected: RuntimeError: Error loading D:\projects\python\MLCourse\03_agentic_ai\data\notes\_broken_encod...

[cleanup] broken demo file removed.
[after cleanup] back to 3 clean Documents.


### 6. Pitfalls worth remembering

**Pitfall - silent skips hide missing knowledge**: with `silent_errors=True`
a corrupt file simply vanishes from your corpus - retrieval will never know
what it cannot find. Prefer loud failures during development; enable skipping
only with a logged audit trail you actually read.

**Pitfall - one loader_cls for many formats**: DirectoryLoader applies ONE
loader class to every glob hit. A `.json` caught by a `*.txt` glob loads as
raw text - silently wrong. Keep globs tight per format, or run several
DirectoryLoaders (one per extension).

**Pro-tip**: the default glob `**/[!.]*` already excludes hidden files -
handy for ignoring editor droppings like `.DS_Store`.

### Takeaway

**DirectoryLoader = glob + loader_cls + recursion over a tree, returning
every child Document with intact per-file provenance. Audit counts per
source, keep globs narrow, and decide your error policy ON PURPOSE.**

### Summary

- Three text files became three Documents via one DirectoryLoader call,
  with `source` metadata naming each origin file.
- Per-source summaries turn bulk loads into auditable inventories.
- The broken-UTF8 experiment showed both policies: raise-by-default versus
  warn-and-skip - and why silent data loss is the dangerous half.
- Cleanup restored the corpus, proving the pipeline is rerunnable.